In [ ]:
#Dataset Download from Kaggle API

!pip install -q kagglehub scikit-learn pandas numpy matplotlib

import kagglehub
import os
import pandas as pd


path = kagglehub.dataset_download("shanegerami/ai-vs-human-text")
print(" Dataset downloaded to:", path)

csv_path = None
for root, dirs, files in os.walk(path):
    for f in files:
        if f.lower().endswith(".csv"):
            csv_path = os.path.join(root, f)

if csv_path is None:
    raise FileNotFoundError("CSV not found!")

print("Found CSV:", csv_path)

df = pd.read_csv(csv_path)

print("\nDataset loaded")
print(df.head())
print("\nClass distribution:  0 (Human), 1(AI generated)")
print(df["generated"].value_counts())

✅ Dataset downloaded to: C:\Users\xhani\.cache\kagglehub\datasets\shanegerami\ai-vs-human-text\versions\1
Found CSV: C:\Users\xhani\.cache\kagglehub\datasets\shanegerami\ai-vs-human-text\versions\1\AI_Human.csv

Dataset loaded
                                                text  generated
0  Cars. Cars have been around since they became ...        0.0
1  Transportation is a large necessity in most co...        0.0
2  "America's love affair with it's vehicles seem...        0.0
3  How often do you ride in a car? Do you drive a...        0.0
4  Cars are a wonderful thing. They are perhaps o...        0.0

Class distribution:  0 (Human), 1(AI generated)
generated
0.0    305797
1.0    181438
Name: count, dtype: int64


In [ ]:
#Balancing the dataset

ai_written=df[df["generated"]== 1].sample(5000, random_state=42)
human_written = df[df["generated"] == 0].sample(5000, random_state=42)

balanced_dataset=pd.concat([ai_written, human_written]).sample(frac=1,random_state=42)

print(balanced_dataset["generated"].value_counts())
print(balanced_dataset.head())


generated
0.0    5000
1.0    5000
Name: count, dtype: int64
                                                     text  generated
269265  Dear Principal,\n\nWe have been hearing quite ...        0.0
327273   Dear [State Senator],\n\nI am writing to expr...        1.0
167398  As high school students, we are constantly bom...        1.0
291678  Hi, I'm a 6th garden and I think zoos ANE Near...        1.0
457139  Sure, jar's my attempt at writing an essay as ...        1.0


In [ ]:
#Train/Test Split

from sklearn.model_selection import train_test_split

X=balanced_dataset["text"]
y=balanced_dataset["generated"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)


In [18]:
#Feature Extraction using text statistics

import numpy as np

def text_extraction(text_statistics):
    new_features=[]
    for text in text_statistics:
        if type(text)== str:
            word=text.split()
            nr_words=len(word)
            nr_chars=len(text)
            unique_words=len(set(word))
            if nr_words:
                avg_word_length=nr_chars/nr_words
            else: 
                avg_word_length=0
            if nr_chars:
                punctuation_ratio= sum(char in".,;:!?'" for char in text) / nr_chars
            else:
                punctuation_ratio=0
            if nr_chars:
                capital_ratio= sum(char.isupper() for char in text) / nr_chars
            else:
                capital_ratio=0
            if nr_chars:
                digit_ratio= sum(char.isdigit() for char in text) / nr_chars
            else:
                digit_ratio=0   
            if nr_words:
                type_token_r=unique_words/nr_words
            else:    
                type_token_r=0

            new_features.append ([nr_words, nr_chars, avg_word_length, punctuation_ratio, capital_ratio, digit_ratio, type_token_r])  

    return np.array(new_features) 

X_train_new_f=text_extraction(X_train)
X_test_new_f=text_extraction(X_test)

In [ ]:
#Decision Tree

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score



depth_testing=[1,2,3,5,10,None]
leaves=[1,2,5,10,20]
criterion="gini"

for d in depth_testing:
    for l in leaves:
        tree = DecisionTreeClassifier(criterion=criterion,max_depth=d,min_samples_leaf=l,random_state=42)

        results = []

        tree.fit(X_train_new_f,y_train)
        preds = tree.predict(X_test_new_f)

        accuracy = accuracy_score(y_test, preds)    
        print(
                f"Criterion={criterion} | "
                f"Depth={d} | "
                f"Leaf={l} | "
                f"Accuracy={accuracy:.4f}"
            )    
        results.append({
                "Criterion": criterion,
                "Max Depth": d,
                "Min Samples Leaf": l,
                "Accuracy": accuracy,
                "Precision": precision_score(y_test, preds),
                "Recall": recall_score(y_test, preds),
                "F1": f1_score(y_test, preds)
                })
                       

#import pandas as pd

#results_df = pd.DataFrame(results)

#print(results_df)


Criterion=gini | Depth=1 | Leaf=1 | Accuracy=0.7505
Criterion=gini | Depth=1 | Leaf=2 | Accuracy=0.7505
Criterion=gini | Depth=1 | Leaf=5 | Accuracy=0.7505
Criterion=gini | Depth=1 | Leaf=10 | Accuracy=0.7505
Criterion=gini | Depth=1 | Leaf=20 | Accuracy=0.7505
Criterion=gini | Depth=2 | Leaf=1 | Accuracy=0.7800
Criterion=gini | Depth=2 | Leaf=2 | Accuracy=0.7800
Criterion=gini | Depth=2 | Leaf=5 | Accuracy=0.7800
Criterion=gini | Depth=2 | Leaf=10 | Accuracy=0.7800
Criterion=gini | Depth=2 | Leaf=20 | Accuracy=0.7800
Criterion=gini | Depth=3 | Leaf=1 | Accuracy=0.7885
Criterion=gini | Depth=3 | Leaf=2 | Accuracy=0.7885
Criterion=gini | Depth=3 | Leaf=5 | Accuracy=0.7885
Criterion=gini | Depth=3 | Leaf=10 | Accuracy=0.7885
Criterion=gini | Depth=3 | Leaf=20 | Accuracy=0.7880
Criterion=gini | Depth=5 | Leaf=1 | Accuracy=0.8470
Criterion=gini | Depth=5 | Leaf=2 | Accuracy=0.8465
Criterion=gini | Depth=5 | Leaf=5 | Accuracy=0.8460
Criterion=gini | Depth=5 | Leaf=10 | Accuracy=0.8495
Crite

In [72]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

knn=KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_new_f, y_train)
preds_raw = knn.predict(X_test_new_f)

print("KNN without scaling accuracy:", accuracy_score(y_test, preds_raw))

KNN without scaling accuracy: 0.773


In [73]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_new_f)
X_test_scaled  = scaler.transform(X_test_new_f)

knn=KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_new_f, y_train)
preds_raw = knn.predict(X_test_new_f)

print("Scaled KNN  accuracy:", accuracy_score(y_test, preds_raw))

Scaled KNN  accuracy: 0.773
